# llm-lab: обучение на Colab T4
Рантайм: GPU (T4). Ключ W&B: Colab Secrets, имя `WANDB_API_KEY`. После обрыва перезапустите ячейки по порядку: обучение продолжится само.

In [ ]:
import torch
assert torch.cuda.is_available(), "Включите GPU: Runtime -> Change runtime type -> T4"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
REPO_URL = "https://github.com/olegzlobin/llm-lab.git"  # заглушка: впишите URL репозитория
CONFIG = "configs/baseline.yaml"
# Алгебра (второй датасет): CONFIG = "configs/algebra/baseline.yaml"
CKPT_DIR = "/content/drive/MyDrive/llm-lab/checkpoints/baseline"  # на Drive, переживает обрывы
# Для алгебры: CKPT_DIR = "/content/drive/MyDrive/llm-lab/checkpoints/algebra_baseline"

import os
if not os.path.exists("/content/llm-lab"):
    !git clone $REPO_URL /content/llm-lab
%cd /content/llm-lab
!pip install -q -e . wandb

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception as e:
    print("Нет WANDB_API_KEY в Colab Secrets, W&B отключён:", e)
    os.environ["WANDB_MODE"] = "disabled"

In [ ]:
# Для другого конфига (например, algebra) prepare_data запускается заново: у каждого датасета свой кэш в data/.
!python scripts/prepare_data.py -c $CONFIG

In [ ]:
# Запуск/продолжение обучения. Чекпойнты на Drive; тот же W&B run подхватывается автоматически.
!python scripts/train.py -c $CONFIG --ckpt-dir $CKPT_DIR